In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm
from tqdm.contrib import tenumerate

# 1. Configuración de Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "microsoft/codebert-base"
dataset_repo = "maddyrucos/code_vulnerability_python"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Clase Dataset Robusta
class VulnerabilityDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=512):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, item):
        codigo = str(self.dataset[item]['func'])
        label = int(self.dataset[item]['target'])

        # Procesamiento de texto
        encoding = self.tokenizer(
            codigo,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt' # Retorna Tensores de PyTorch
        )

        # Squeeze para eliminar la dimensión extra del batch que añade el tokenizer por defecto
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# 3. Carga de Modelos y Datos
print("Cargando CodeBERT y Dataset...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

# Cargar dataset completo
dataset = load_dataset(dataset_repo, split='train')

# Separar en entrenamiento y validación
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset_hf = dataset_split['train']
val_dataset_hf = dataset_split['test']

Cargando CodeBERT y Dataset...


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/384 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/122k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/848 [00:00<?, ? examples/s]

In [ ]:


# 4. Definición del Modelo (Arquitectura de Espacio Latente)
class VulnerabilityModel(nn.Module):
    def __init__(self, codebert):
        super(VulnerabilityModel, self).__init__()
        self.codebert = codebert
        # Capa densa para clasificar el espacio latente (768) en 2 clases
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        # Representación en el espacio latente (Vector del token [CLS])
        latent_vector = outputs.last_hidden_state[:, 0, :]

        #Logica del clasificador binario
        logits = self.classifier(latent_vector)

        #En otra libreta de colab


        return logits, latent_vector

# Inicializar componentes
model = VulnerabilityModel(base_model).to(device)
train_dataset = VulnerabilityDataset(train_dataset_hf, tokenizer)
val_dataset = VulnerabilityDataset(val_dataset_hf, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

optimizer = AdamW(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
# 5. Bucle de Entrenamiento
epochs = 15
print(f"Entrenando en: {device}")
model.train()
for epoch in range(epochs):

    #Inicializar metricas
    loss_epoch = 0.0
    rend_epoch = 0.0
    n_items = 0

    for i, batch in tenumerate(train_loader):
        # Mover datos a GPU/CPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Paso hacia adelante
        optimizer.zero_grad()
        logits, _ = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        # Obtener labels del modelo
        labels_pred = np.argmax(logits.cpu().detach().numpy())

        # Optimización
        loss.backward()
        optimizer.step()

        #Obtener etiquetas del modelo
        labels_pred = logits.cpu().detach().numpy()
        predictions = np.argmax(labels_pred, axis = 1)
        rend = np.sum(predictions == labels.cpu().detach().numpy())
        n_items += predictions.shape[0]

        #Acumular loss y rendimiento
        loss_epoch += float(loss.item())
        rend_epoch += rend

    #Imprimir detalles del entrenamiento
    print(f"Epoca: {epoch} | Pérdida: {loss_epoch/len(train_loader)} | Rendimiento: {rend_epoch/n_items}")

    # VALIDACIÓN

    model.eval()

    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits, _ = model(input_ids, attention_mask)

            preds = torch.argmax(logits, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds)

    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Validation F1: {val_f1:.4f}")

    model.train()


print("¡Entrenamiento finalizado con éxito!")

torch.save(
    model.state_dict(),
    '/content/drive/MyDrive/Weights/model_weights_binary.pth',
    _use_new_zipfile_serialization=False

)

print("Pesos guardados correctamente")

from google.colab import files

# files.download('model_weights_binary.pth')

Entrenando en: cuda


  0%|          | 0/43 [00:00<?, ?it/s]

Epoca: 0 | Pérdida: 0.6390718734541605 | Rendimiento: 0.6460176991150443
Validation Accuracy: 0.7471
Validation F1: 0.7571


  0%|          | 0/43 [00:00<?, ?it/s]

Epoca: 1 | Pérdida: 0.46894638482914414 | Rendimiento: 0.7935103244837758
Validation Accuracy: 0.8000
Validation F1: 0.8172


...

Epoca: 12 | Pérdida: 0.03045542766664957 | Rendimiento: 0.9926253687315634
Validation Accuracy: 0.9294
Validation F1: 0.9259


  0%|          | 0/43 [00:00<?, ?it/s]

Epoca: 13 | Pérdida: 0.024930643683927525 | Rendimiento: 0.9926253687315634
Validation Accuracy: 0.9294
Validation F1: 0.9250


  0%|          | 0/43 [00:00<?, ?it/s]

Epoca: 14 | Pérdida: 0.025293558695201956 | Rendimiento: 0.9911504424778761
Validation Accuracy: 0.9294
Validation F1: 0.9231
¡Entrenamiento finalizado con éxito!
Pesos guardados correctamente


In [ ]:
torch.save(
    model.state_dict(),
    '/content/drive/MyDrive/Weights/model_weights_binary.pth',
    _use_new_zipfile_serialization=False
)

checkpoint = torch.load(
    '/content/drive/MyDrive/Weights/model_weights_binary.pth'
)

print(checkpoint['classifier.weight'].shape)
print("Pesos guardados correctamente")


torch.Size([2, 768])
Pesos guardados correctamente


In [ ]:
import torch

obj = torch.load(
    '/content/drive/MyDrive/Weights/model_weights_binary.pth',
    map_location='cpu'
)

print(type(obj))

<class 'collections.OrderedDict'>


In [ ]:
#Obtener etiquetas del modelo
labels_pred = logits.cpu().detach().numpy()
print("Salida sin filtro: ", labels_pred)
predictions = np.argmax(labels_pred, axis = 1)
print("Etiquetas del modelo: " + str(predictions))
print(labels.cpu())
print(predictions == labels.cpu().detach().numpy())
rend = np.sum(predictions == labels.cpu().detach().numpy())/predictions.shape[0]
print(rend)

Salida sin filtro:  [[ 4.57134   -4.0492625]
 [ 2.9308057 -2.5833085]
 [-3.5791972  4.6734695]
 [-2.9970765  3.7234564]
 [ 4.0950837 -3.7804065]
 [ 4.438522  -4.105425 ]
 [ 4.4642105 -3.969572 ]
 [-3.6265714  4.7573705]
 [ 4.520279  -4.0512114]
 [-3.487891   4.5293345]]
Etiquetas del modelo: [0 0 1 1 0 0 0 1 0 1]
tensor([0, 0, 1, 1, 0, 0, 0, 1, 0, 1])
[ True  True  True  True  True  True  True  True  True  True]
1.0


In [ ]:
# PRUEBA MANUAL DE CÓDIGO}

def predecir_codigo(texto):

    model.eval()

    encoding = tokenizer(
        texto,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():

        logits, _ = model(input_ids, attention_mask)

        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

        confianza = probs[0][pred].item()

    clases = {
        0: "Seguro",
        1: "Malicioso"
    }

    print("=" * 50)
    print("Resultado:")
    print(f"Clasificación: {clases[pred]}")
    print(f"Confianza: {confianza:.4f}")
    print("=" * 50)

# EJEMPLO DE USO

codigo ="""
import os
import re
import secrets
from typing import Optional

def procesar_nombre_archivo(nombre_archivo: str) -> Optional[str]:
    patron = re.compile(r'^[a-zA-Z0-9._-]+$')
    if not patron.match(nombre_archivo):
        raise ValueError("Nombre de archivo inválido detectado.")

    return nombre_archivo

def generar_token_sesion() -> str:
    return secrets.token_hex(32)

def main() -> None:
    # 1. Gestión de credenciales (nunca ponerlas en texto plano en el código)
    # Ejemplo de uso de variables de entorno:
    clave_api = os.getenv('CLAVE_API_SECRETA', 'valor_por_defecto_inseguro')

    if clave_api == 'valor_por_defecto_inseguro':
        # En producción, esto debería ser un error crítico
        print("Advertencia: No se ha configurado la variable de entorno para la API.")

    # 2. Validación de entrada
    entrada_usuario = "reporte_usuario_valido.txt"
    archivo_seguro = procesar_nombre_archivo(entrada_usuario)
    print(f"Archivo procesado con éxito: {archivo_seguro}")

    # 3. Generación de valores aleatorios seguros
    token = generar_token_sesion()
    print("Token de sesión generado de forma segura.")

if __name__ == "__main__":
    main()"""



predecir_codigo(codigo)

Resultado:
Clasificación: Seguro
Confianza: 0.9998
